## CUSTOMER ANALYTICS and RFM ANALYSIS
**By the end of this notebook we can answer questions like-**
1. Who are our most valuable customers ? 
2. Who purchases most frequently ? 
3. Who spends the most ? 
4. Who is becoming inactive ? 
5. Which customers should marketing target ? 
6. How can we segment customers before applying machine learning ? 

In [1]:
import warnings 
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns 

In [ ]:

project_root = Path.cwd().parent.parent
data_dir = project_root / "datasets" / "Olist"


In [3]:
orders = pd.read_csv(data_dir/"olist_orders_dataset.csv",
                     parse_dates=["order_purchase_timestamp"]
                     )
payments = pd.read_csv(
    data_dir/"olist_order_payments_dataset.csv"
)
customers = pd.read_csv(data_dir/"olist_customers_dataset.csv")

In [4]:
# creating a customer master table 
customer_df = (
    customers
    .merge(
        orders,
        on="customer_id",
        how="left"
    )
    .merge(payments,
           on="order_id",
           how="left")
)
customer_df.head(10)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,payment_installments,payment_value
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,00e7ee1b050b8499577073aeb2a297a1,delivered,2017-05-16 15:05:35,2017-05-16 15:22:12,2017-05-23 10:47:57,2017-05-25 10:35:35,2017-06-05 00:00:00,1.0,credit_card,2.0,146.87
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,29150127e6685892b6eab3eec79f59c7,delivered,2018-01-12 20:48:24,2018-01-12 20:58:32,2018-01-15 17:14:59,2018-01-29 12:41:19,2018-02-06 00:00:00,1.0,credit_card,8.0,335.48
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,b2059ed67ce144a36e2aa97d2c9e9ad2,delivered,2018-05-19 16:07:45,2018-05-20 16:19:10,2018-06-11 14:31:00,2018-06-14 17:58:51,2018-06-13 00:00:00,1.0,credit_card,7.0,157.73
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,951670f92359f4fe4a63112aa7306eba,delivered,2018-03-13 16:06:38,2018-03-13 17:29:19,2018-03-27 23:22:42,2018-03-28 16:04:25,2018-04-10 00:00:00,1.0,credit_card,1.0,173.30
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,6b7d50bd145f6fc7f33cebabd7e49d0f,delivered,2018-07-29 09:51:30,2018-07-29 10:10:09,2018-07-30 15:16:00,2018-08-09 20:55:48,2018-08-15 00:00:00,1.0,credit_card,8.0,252.25
5,879864dab9bc3047522c92c82e1212b8,4c93744516667ad3b8f1fb645a3116a4,89254,jaragua do sul,SC,5741ea1f91b5fbab2bd2dc653a5b5099,delivered,2017-09-14 18:14:31,2017-09-14 18:25:11,2017-09-18 21:27:40,2017-09-28 17:32:43,2017-10-04 00:00:00,1.0,debit_card,1.0,282.21
6,fd826e7cf63160e536e0908c76c3f441,addec96d2e059c80c30fe6871d30d177,4534,sao paulo,SP,36e694cf4cbc2a4803200c35e84abdc4,delivered,2018-02-19 14:38:35,2018-02-19 14:50:37,2018-02-20 00:03:39,2018-02-20 16:25:51,2018-03-05 00:00:00,1.0,credit_card,1.0,22.77
7,5e274e7a0c3809e14aba7ad5aae0d407,57b2a98a409812fe9618067b6b8ebe4f,35182,timoteo,MG,1093c8304c7a003280dd34598194913d,delivered,2017-11-16 19:29:02,2017-11-16 19:55:41,2017-11-22 16:46:33,2017-11-27 12:44:36,2017-12-08 00:00:00,1.0,credit_card,3.0,36.01
8,5adf08e34b2e993982a47070956c5c65,1175e95fb47ddff9de6b2b06188f7e0d,81560,curitiba,PR,1ebeea841c590e86a14a0d7a48e7d062,delivered,2018-01-18 12:35:44,2018-01-18 12:56:32,2018-01-18 23:25:35,2018-01-26 15:17:57,2018-02-20 00:00:00,1.0,debit_card,1.0,39.10
9,4b7139f34592b3a31687243a302fa75b,9afe194fb833f79e300e37e580171f22,30575,belo horizonte,MG,7433cbcc783205509d66a5260da5b574,delivered,2018-01-08 11:22:34,2018-01-08 11:35:27,2018-01-11 01:00:40,2018-01-13 14:51:55,2018-02-05 00:00:00,1.0,credit_card,1.0,122.47


In [5]:
# total customer spend 
customer_spend = (
    customer_df
    .groupby("customer_unique_id")
    .agg(
        TotalSpend=("payment_value","sum")
    )
    .sort_values(
        "TotalSpend",
        ascending=False
    ).reset_index()
)
customer_spend.head(10)

,customer_unique_id,TotalSpend
0,0a0a92112bd4c708ca5fde585afaa872,13664.08
1,46450c74a0d8c5ca9395da1daac6c120,9553.02
2,da122df9eeddfedc1dc1f5349a1a690c,7571.63
3,763c8b1c9c68a0229c42c9fc6f662b93,7274.88
4,dc4802a71eae9be1dd28f5d788ceb526,6929.31
5,459bef486812aa25204be022145caa62,6922.21
6,ff4159b92c40ebe40454e3e6a7c35ed6,6726.66
7,4007669dec559734d6f53e029e360987,6081.54
8,5d0a2980b292d049061542014e8960bf,4809.44
9,eebb5dda148d3893cdaf5b5ca3040ccb,4764.34


In [6]:
fig = px.histogram(
    customer_spend,
    x="TotalSpend",
    nbins=60,
    title="Customer Spending Distribution",
    template="plotly_white"
)
fig.show()

In [11]:
# top 20 customers 
top20 = customer_spend.head(20)
fig = px.bar(
    top20,
    x="customer_unique_id",
    y="TotalSpend",
    title = "Top 20 customers by revenue",
    template="plotly_white"
)
fig.update_xaxes(showticklabels=False)
fig.show()

In [12]:
# purchase frequency 
purchase_frequency = (
    customer_df
    .groupby("customer_unique_id")
    .agg(
        Orders=("order_id","nunique")
    )
    .sort_values(
        "Orders",
        ascending=False
    )
    .reset_index()
)
purchase_frequency.head(10)

,customer_unique_id,Orders
0,8d50f5eadf50201ccdcedfb9e2ac8455,17
1,3e43e6105506432c953e165fb2acf44c,9
2,1b6c7548a2a1f9037c1fd3ddfed95f33,7
3,6469f99c1f9dfae7733b25662e7f1782,7
4,ca77025e7201e3b30c44b472ff346268,7
5,47c1a3033b8b77b3ab6e109eb4d5fdf3,6
6,12f5d6e1cbf93dafd9dcc19095df0b3d,6
7,63cfc61cee11cbe306bff5857d00bfe4,6
8,dc813062e0fc23409cd255f7f53c7074,6
9,de34b16117594161a6a89c50b289d35a,6


In [13]:
fig = px.histogram(
    purchase_frequency,
    x="Orders",
    nbins=25,
    title="Purchase frequency distribution",
    template="plotly_white"
)
fig.show()

In [14]:
# customer recency 
snapshot_date=(
    customer_df["order_purchase_timestamp"].max()
    + pd.Timedelta(days=1)
)

In [16]:
recency = (
    customer_df
    .groupby("customer_unique_id")
    .agg(
        LastPurchase=(
            "order_purchase_timestamp",
            "max"
        )
    )
)
recency["Recency"]=(
    snapshot_date-recency["LastPurchase"]
).dt.days
recency.reset_index(inplace=True)
recency.head()

,customer_unique_id,LastPurchase,Recency
0,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,161
1,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,164
2,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,586
3,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,370
4,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,337


In [18]:
fig=px.histogram(
    recency,
    x="Recency",
    nbins=40,
    title="Customer Recency Distribution",
    template="plotly_white"
)
fig.show()

In [ ]:
# rfm table 
rfm = (
    customer_df
    .groupby("customer_unique_id")
    .agg({
        "order_purchase_timestamp":"max",
        "order_id":"nunique",
        "payment_value":"sum"
    })
)
rfm.columns=[
    "LastPurchase",
    "Frequency",
    "Monetary"
]
rfm["Recency"]=(
    snapshot_date - rfm["LastPurchase"]
).dt.days

rfm = rfm[
    ["Recency","Frequency","Monetary"]
]
rfm.head()

,Recency,Frequency,Monetary
customer_unique_id,,,
0000366f3b9a7992bf8c76cfdf3221e2,161,1,141.90
0000b849f77a49e4a4ce2b2a4ca5be3f,164,1,27.19
0000f46a3911fa3c0805444483337064,586,1,86.22
0000f6ccb0745a6a4b88665a16c9f078,370,1,43.62
0004aac84e0df4da2b147fca70cf8255,337,1,196.89


In [20]:
# rfm scores 
rfm["R"]=pd.qcut(
    rfm["Recency"],
    5,
    labels=[5,4,3,2,1]
)
rfm["F"]=pd.qcut(
    rfm["Frequency"].rank(method="first"),
    5,
    labels=[1,2,3,4,5]
)
rfm["M"]=pd.qcut(
    rfm["Monetary"],
    5,
    labels=[1,2,3,4,5]
)
rfm.head()


,Recency,Frequency,Monetary,R,F,M
customer_unique_id,,,,,,
0000366f3b9a7992bf8c76cfdf3221e2,161,1,141.90,4,1,4
0000b849f77a49e4a4ce2b2a4ca5be3f,164,1,27.19,4,1,1
0000f46a3911fa3c0805444483337064,586,1,86.22,1,1,2
0000f6ccb0745a6a4b88665a16c9f078,370,1,43.62,2,1,1
0004aac84e0df4da2b147fca70cf8255,337,1,196.89,2,1,4


In [21]:
# overall scores 
rfm["RFM_score"]=(
    rfm["R"].astype(str)
    +
    rfm["F"].astype(str)
    +
    rfm["M"].astype(str)
)
rfm.head()

,Recency,Frequency,Monetary,R,F,M,RFM_score
customer_unique_id,,,,,,,
0000366f3b9a7992bf8c76cfdf3221e2,161,1,141.90,4,1,4,414
0000b849f77a49e4a4ce2b2a4ca5be3f,164,1,27.19,4,1,1,411
0000f46a3911fa3c0805444483337064,586,1,86.22,1,1,2,112
0000f6ccb0745a6a4b88665a16c9f078,370,1,43.62,2,1,1,211
0004aac84e0df4da2b147fca70cf8255,337,1,196.89,2,1,4,214


In [22]:
# now defining customer segment 
def segment(row):
    if row["R"]>=4 and row["F"]>=4:
        return "Champions"
    elif row["R"]>=3 and row["F"]>=3:
        return "Loyal Customer"
    elif row["R"]>=4:
        return "Potential Loyalies"
    elif row["R"]<=2 and row["F"]>=4:
        return "At Risk"
    else:
        return "Others"
    
rfm["Segment"] = rfm.apply(
    segment,
    axis=1
)
rfm.head()

,Recency,Frequency,Monetary,R,F,M,RFM_score,Segment
customer_unique_id,,,,,,,,
0000366f3b9a7992bf8c76cfdf3221e2,161,1,141.90,4,1,4,414,Potential Loyalies
0000b849f77a49e4a4ce2b2a4ca5be3f,164,1,27.19,4,1,1,411,Potential Loyalies
0000f46a3911fa3c0805444483337064,586,1,86.22,1,1,2,112,Others
0000f6ccb0745a6a4b88665a16c9f078,370,1,43.62,2,1,1,211,Others
0004aac84e0df4da2b147fca70cf8255,337,1,196.89,2,1,4,214,Others


In [23]:
# visualization 
segment_count=(
    rfm["Segment"]
    .value_counts()
    .reset_index()
)

segment_count.columns=[
    "Segment",
    "Customers"
]

fig = px.pie(
    segment_count,
    names="Segment",
    values="Customers",
    title="Customer Segments",
    template="plotly_white"
)
fig.show()

In [24]:
# segment wise revenue 
segment_revenue = (
    rfm
    .groupby("Segment")
    .agg(
        Revenue=("Monetary","sum"),
        Customers=("Monetary","count")
    )
    .reset_index()
)
segment_revenue

,Segment,Revenue,Customers
0,At Risk,2621214.41,15285
1,Champions,2773648.20,15453
2,Loyal Customer,3111244.94,19237
3,Others,4959026.53,30672
4,Potential Loyalies,2543738.04,15449


In [25]:
fig = px.bar(
    segment_revenue,
    x="Segment",
    y="Revenue",
    color="Segment",
    title="Revenue contribution by Segment",
    template="plotly_white"
)
fig.show()

# Business Insights

The customer analytics and RFM analysis provide valuable insights into customer purchasing behavior, spending patterns, and engagement levels across the Olist marketplace.

## 1. Customer Spending Behavior

- Customer spending follows a highly skewed distribution, where a small percentage of customers contribute a disproportionately large share of total revenue.
- The highest-spending customer has generated over **R$13,600**, while the majority of customers contribute significantly smaller amounts.
- This indicates that the marketplace follows the **Pareto Principle (80/20 rule)**, where a relatively small group of customers generates a large proportion of total sales.

---

## 2. Purchase Frequency

- Most customers have placed only a single order throughout the observation period.
- The most active customer placed **17 orders**, while only a very small percentage of customers made multiple purchases.
- This suggests that customer acquisition is strong, but repeat purchasing remains relatively low.
- Increasing repeat purchases should be a key business objective.

---

## 3. Customer Recency

- Customer recency varies considerably across the customer base.
- Many customers have not made a purchase for several months, indicating potential customer churn.
- Recently active customers represent valuable opportunities for upselling and cross-selling, while inactive customers may benefit from re-engagement campaigns.

---

## 4. Revenue Contribution by Customer Segment

The RFM segmentation highlights distinct customer groups with different business value.

- **Loyal Customers** contribute the highest overall revenue (**~R$3.11 million**), making them the most valuable segment.
- **Champions** generate approximately **R$2.77 million** while maintaining strong purchasing behavior and recent activity.
- **Potential Loyalists** contribute approximately **R$2.54 million** and represent customers who can be nurtured into long-term loyal buyers.
- **At-Risk Customers** still contribute over **R$2.62 million**, indicating that losing these customers could significantly impact future revenue.
- The **Others** segment represents the largest customer group but consists primarily of low-frequency and lower-value customers.

---

## 5. Customer Segmentation

The RFM model successfully categorizes customers into meaningful business segments, enabling targeted customer relationship management instead of applying identical marketing strategies to every customer.

These segments provide a foundation for personalized campaigns and future predictive models such as Customer Lifetime Value (CLV).

---

# Strategic Business Recommendations

### Reward Champions

Champions are among the highest-value customers and should receive exclusive rewards, early access to products, premium support, and loyalty incentives to maximize retention.

---

### Strengthen Loyal Customers

Loyal customers already contribute the highest revenue. Personalized recommendations, membership benefits, and targeted promotions can further increase purchase frequency and lifetime value.

---

### Convert Potential Loyalists

Potential Loyalists are highly engaged customers with strong growth potential. Timely promotions, personalized product recommendations, and follow-up campaigns can encourage them to become long-term loyal customers.

---

### Re-engage At-Risk Customers

At-Risk customers still represent a significant share of total revenue. Win-back campaigns, targeted discounts, and reminder emails should be implemented before these customers become inactive permanently.

---

### Improve Customer Retention

Since the majority of customers place only a single order, improving customer retention should become a strategic priority. Increasing repeat purchase rates is likely to have a greater long-term impact than continuously investing only in customer acquisition.

---

# Conclusion

The RFM analysis demonstrates that customer behavior varies significantly across the marketplace. While a relatively small group of customers contributes the majority of revenue, a large portion of customers purchase only once and gradually become inactive.

Segmenting customers based on Recency, Frequency, and Monetary value enables the business to develop targeted retention strategies, optimize marketing spend, and improve customer lifetime value. These customer segments will serve as the foundation for subsequent machine learning tasks, including customer segmentation refinement and Customer Lifetime Value prediction.

We can later create more detailed customer segmentation like 
1.Champions
2.Loyal Customers
3.Potential Loyalists
4.New Customers
5.Promising
6.Need Attention
7.About to Sleep
8.At Risk
9.Can't Lose Them
10.Hibernating
11.Lost